# SWASTHYAGUIDE 2.0 — 02. Preprocessing & Baselines

**Stage 2.** Builds a leakage-safe preprocessing pipeline, trains three un-tuned
baselines, and reports overall plus cohort-wise metrics.

**Scope disclaimer.** Research prototype for health **risk assessment**. Not a
medical device, not a diagnostic system, not validated for clinical use.

**Data rule.** `data/heart_disease_uci.csv` is read-only. Nothing here writes to
it; the checksum is re-verified at the end of this notebook.

## 1. Imports

The pipeline logic lives in `src/preprocessing.py` rather than inline here. This
notebook imports and exercises it, so what is demonstrated is exactly what ships.

In [ ]:
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

PROJECT_ROOT = Path.cwd()
while not (PROJECT_ROOT / "src" / "preprocessing.py").exists():
    if PROJECT_ROOT.parent == PROJECT_ROOT:
        raise FileNotFoundError("src/preprocessing.py not found")
    PROJECT_ROOT = PROJECT_ROOT.parent
sys.path.insert(0, str(PROJECT_ROOT))

from src import preprocessing as prep
from src.evaluate import (
    compute_metrics,
    confusion_frame,
    format_cohort_table,
    format_metric_table,
    per_cohort_metrics,
)
from src.train import describe_split, make_cohort_stratified_split

RANDOM_SEED = prep.RANDOM_SEED
np.random.seed(RANDOM_SEED)
sns.set_theme(style="whitegrid", context="notebook")
pd.set_option("display.width", 200)
pd.set_option("display.max_columns", 60)

print("seed:", RANDOM_SEED)
print("data:", prep.DATA_PATH)

## 2. Load data and verify the raw file is untouched

`load_dataset()` raises if the SHA-256 does not match the recorded hash, so an
accidental edit to the source CSV cannot pass silently.

In [ ]:
print("expected sha256:", prep.EXPECTED_SHA256)
print("actual   sha256:", prep.file_sha256())

df = prep.build_analysis_frame()
X, y, cohorts = prep.split_xy(df)

print("rows:", len(df), "| features:", len(prep.FEATURE_COLUMNS))
print("excluded from features:", prep.EXCLUDED_FROM_FEATURES)
print("class balance:", y.value_counts().sort_index().to_dict())
print("positive rate:", round(float(y.mean()), 4))
df.head()

## 3. Target definition

`num == 0 -> 0` (no disease), `num > 0 -> 1` (disease). Applied in memory only;
`num` is preserved so the ordinal target remains available for later comparison.

In [ ]:
pd.DataFrame({
    "num (raw ordinal)": df["num"].value_counts().sort_index(),
    "target (binary)": df["target"].value_counts().sort_index(),
})

assert set(df.loc[df["num"] == 0, "target"].unique()) == {0}
assert set(df.loc[df["num"] > 0, "target"].unique()) == {1}
print("rule verified: num == 0 -> 0, num > 0 -> 1")

## 4. Preprocessing decisions

Four decisions drive the pipeline. Each is justified by something measurable in
this dataset rather than by convention.

### 4.1 Invalid physiological sentinels -> NaN

A zero resting blood pressure, a zero serum cholesterol, or a *negative* ST
depression cannot be real measurements. Here they encode "not recorded".

| Column | Rule | Rows affected (train) |
|--------|------|------------------------|
| `chol` | `== 0` -> NaN | 139 |
| `trestbps` | `== 0` -> NaN | 1 |
| `oldpeak` | `< 0` -> NaN | 10 |

This runs inside the pipeline (`InvalidValueMasker`), so unseen inference data is
cleaned by the same rule. It is a class, not a closure, so the fitted pipeline can
be pickled.

In [ ]:
rows = []
for col, rule in prep.INVALID_VALUE_RULES.items():
    values = X[col].to_numpy(dtype=float)
    hits = int(np.sum(values == rule["value"])) if rule["op"] == "==" else int(np.sum(values < rule["value"]))
    rows.append({
        "column": col,
        "rule": f"{rule['op']} {rule['value']:.0f} -> NaN",
        "rows_affected": hits,
        "reason": rule["reason"],
    })
pd.DataFrame(rows)

### 4.2 `ca`: ordinal numeric, inspected before deciding

`ca` is 66.4% missing, and 99% of its non-missing values come from Cleveland
alone. Both facts below were measured, not assumed.

In [ ]:
print("ca values:", sorted(df["ca"].dropna().unique().tolist()))
print("all integral?", bool(np.all(np.equal(np.mod(df["ca"].dropna(), 1), 0))))

ca_vs_target = df.groupby("ca")["target"].agg(n="count", p_disease="mean")
ca_vs_target["p_disease"] = (ca_vs_target["p_disease"] * 100).round(2)
ca_vs_target

In [ ]:
from scipy.stats import spearmanr

sub = df.dropna(subset=["ca"])
rho, pval = spearmanr(sub["ca"], sub["target"])
print(f"Spearman(ca, target): rho = {rho:.3f}, p = {pval:.2g}")

rates = ca_vs_target["p_disease"].to_numpy()
print("monotonic non-decreasing in ca?", bool(np.all(np.diff(rates) >= 0)))
print("successive differences:", np.round(np.diff(rates), 2))

fig, ax = plt.subplots(figsize=(6, 3.5))
ax = ca_vs_target["p_disease"].plot.bar(ax=ax, color="#4c72b0")
ax.set_title("Disease prevalence by number of narrowed vessels (ca)")
ax.set_xlabel("ca")
ax.set_ylabel("% disease (num > 0)")
for container in ax.containers:
    ax.bar_label(container, fmt="%.1f%%")
plt.tight_layout()
plt.show()

**Decision: treat `ca` as ordinal numeric, not one-hot categorised.** It is a
genuine count, its values are integral `{0,1,2,3}`, prevalence is monotonic in it,
and Spearman rho = 0.48 confirms a real ordering signal.

**But median imputation alone would be biased here.** The training median of `ca`
is 0.0, so the 66.4% of rows with missing `ca` get filled with "0 narrowed
vessels" — a healthy-looking value — even though rows with missing `ca` are more
likely to be diseased (59.7% vs 46.6%). A missingness indicator is therefore
mandatory, not decorative: it lets the model learn the elevated base rate that
comes with an unrecorded `ca`.

In [ ]:
ca_missingness = df.assign(ca_missing=df["ca"].isna()).groupby("ca_missing")["target"].agg(
    n="count", p_disease="mean"
)
ca_missingness["p_disease"] = (ca_missingness["p_disease"] * 100).round(2)
ca_missingness

print("ca missing % by cohort:")
print(df.groupby("dataset")["ca"].apply(lambda s: round(s.isna().mean() * 100, 1)).to_string())

### 4.3 Boolean text columns -> real numeric

`fbs` and `exang` arrive as the strings `"True"`/`"False"`. They become floats
1.0 / 0.0 with NaN preserved, so a genuinely unrecorded flag is distinguishable
from a recorded `False`. Unrecognised text becomes NaN rather than being coerced.

In [ ]:
raw_cols = ["fbs", "exang"]
raw_df = pd.read_csv(prep.DATA_PATH)

comparison = pd.DataFrame({
    "raw_dtype": [raw_df[c].dtype for c in raw_cols],
    "raw_tokens": [sorted(raw_df[c].dropna().unique().tolist()) for c in raw_cols],
    "converted_dtype": [str(X[c].dtype) for c in raw_cols],
    "converted_values": [sorted(X[c].dropna().unique().tolist()) for c in raw_cols],
    "n_missing": [int(X[c].isna().sum()) for c in raw_cols],
})
comparison

### 4.4 Missing-value strategy — nothing is dropped

Only 299 of 920 rows are complete, so deleting incomplete rows would discard two
thirds of the data. Instead:

* **Numeric** — median imputation fitted on training data only, plus a
  `missingindicator_<col>` column for every column that has any missing value in
  the fitted data.
* **Categorical** — an explicit `Missing` level, then one-hot encoding with
  `handle_unknown="ignore"` so an unseen level at inference yields a zero row
  instead of raising.
* **Row count** is never reduced. Verified in section 8.

In [ ]:
missing_report = pd.DataFrame({
    "dtype": X.dtypes.astype(str),
    "n_missing": X.isna().sum(),
    "pct_missing": (X.isna().sum() / len(X) * 100).round(2),
})
missing_report.sort_values("pct_missing", ascending=False)

print("complete rows:", int(X.notna().all(axis=1).sum()), "of", len(X),
      "-> dropping them is not viable")

## 5. Final feature list

13 raw features go in. The fitted pipeline expands them to more columns via
missingness indicators and one-hot dummies.

In [ ]:
feature_table = pd.DataFrame(
    {
        "order": range(1, len(prep.FEATURE_COLUMNS) + 1),
        "feature": prep.FEATURE_COLUMNS,
        "branch": [
            "numeric" if c in prep.NUMERIC_FEATURES else
            ("numeric (binary flag)" if c in prep.BOOLEAN_SOURCE_COLS else "categorical")
            for c in prep.FEATURE_COLUMNS
        ],
        "pct_missing": [(X[c].isna().mean() * 100).round(2) for c in prep.FEATURE_COLUMNS],
        "description": [prep.FEATURE_DOC[c] for c in prep.FEATURE_COLUMNS],
    }
)
feature_table

In [ ]:
print("Never used as features:")
for col in prep.EXCLUDED_FROM_FEATURES:
    reason = {
        "id": "row counter 1..920, no clinical signal",
        "dataset": "source cohort metadata, used only for split + reporting",
        "num": "the target itself",
        "target": "derived label",
    }[col]
    print(f"  {col:9s} - {reason}")

## 6. Train/test methodology

`train_test_split(test_size=0.2, random_state=42, stratify=cohort x label)`.

Stratifying on the **joint** cohort x label key rather than on the label alone is
deliberate. Disease prevalence ranges from 36.2% (Hungary) to 93.5%
(Switzerland), so plain stratification could leave a cohort missing from one
partition or badly unbalanced within it. The joint key guarantees every cohort
appears in both partitions with a comparable class balance.

In [ ]:
TEST_SIZE = 0.2
train_idx, test_idx = make_cohort_stratified_split(df, TEST_SIZE, RANDOM_SEED)

X_train, X_test = X.iloc[train_idx], X.iloc[test_idx]
y_train, y_test = y.iloc[train_idx].to_numpy(), y.iloc[test_idx].to_numpy()
cohorts_test = cohorts.iloc[test_idx]

split_summary = describe_split(df, train_idx, test_idx)
split_summary

print(f"train {len(X_train)} rows (prevalence {y_train.mean():.4f}) | "
      f"test {len(X_test)} rows (prevalence {y_test.mean():.4f})")
print("disjoint:", len(set(train_idx) & set(test_idx)) == 0)

## 7. Train three un-tuned baselines

Logistic Regression, Random Forest, XGBoost — default-ish hyper-parameters, no
search. Every model shares ONE preprocessing pipeline, so the comparison reflects
the model rather than the preprocessing.

In [ ]:
print(prep.CA_DECISION)

In [ ]:
metrics_by_model = {}
cohort_metrics_by_model = {}
confusion_by_model = {}
probabilities = {}
feature_names = None

for name, factory in prep.BASELINE_MODELS.items():
    pipeline = prep.build_pipeline(factory(random_state=RANDOM_SEED))
    pipeline.fit(X_train, y_train)   # TRAIN ONLY -> no test-set leakage
    probabilities[name] = pipeline.predict_proba(X_test)[:, 1]
    metrics_by_model[name] = compute_metrics(y_test, probabilities[name])
    cohort_metrics_by_model[name] = per_cohort_metrics(
        y_test, probabilities[name], cohorts_test.reset_index(drop=True)
    )
    confusion_by_model[name] = confusion_frame(y_test, probabilities[name])
    if feature_names is None:
        feature_names = prep.transformed_feature_names(pipeline)

pd.Series(feature_names).to_frame("transformed feature").T

In [ ]:
fig, ax = plt.subplots(figsize=(8, 3.5))
scaled = pd.Series([0.0] + [0] * (len(feature_names) - 1))
for name, probs in probabilities.items():
    ax.plot(pd.Series(probs).rank(method="first").to_numpy(), probs, alpha=0.6, label=name)
ax.plot([0, len(y_test)], [0, 1], linestyle="--", color="grey", linewidth=1, label="random")
ax.set_xlabel("test cases, ordered by predicted risk")
ax.set_ylabel("predicted probability of disease")
ax.set_title("Predicted probabilities on the test set")
ax.legend()
plt.tight_layout()
plt.show()

## 8. Leakage checks

Asserted rather than assumed. Each check either passes or the notebook stops.

In [ ]:
fitted = prep.build_pipeline(prep.make_logistic_regression())
fitted.fit(X_train, y_train)
num_pipe = fitted.named_steps["preprocess"].named_transformers_["numeric"]
imputer = num_pipe.named_steps["impute_median"]
scaler = num_pipe.named_steps["scale"]
masker = num_pipe.named_steps["mask_invalid_values"]

num_cols = list(prep.NUMERIC_FEATURES) + list(prep.BOOLEAN_SOURCE_COLS)

# Fitted medians must equal medians computed from TRAIN rows only.
train_masked = masker.transform(X_train[num_cols].to_numpy(dtype=float))
expected_medians = np.nanmedian(train_masked, axis=0)
assert np.allclose(imputer.statistics_, expected_medians)
print("[PASS] imputer medians == train-only medians")

# Scaler statistics must NOT equal full-dataset statistics.
full_masked = masker.transform(X[num_cols].to_numpy(dtype=float))
full_means = np.nanmean(full_masked[:, : len(prep.NUMERIC_FEATURES)], axis=0)
assert not np.allclose(scaler.mean_[: len(prep.NUMERIC_FEATURES)], full_means)
print("[PASS] scaler statistics are train-only")

# No rows dropped anywhere in the pipeline.
assert fitted.predict_proba(X_test).shape[0] == len(X_test)
print(f"[PASS] pipeline preserves row count ({len(X_test)} in, {len(X_test)} out)")

# Sentinels actually become NaN inside the pipeline.
probe = pd.DataFrame([{
    "age": 50, "trestbps": 0, "chol": 0, "thalch": 150, "oldpeak": -2.6, "ca": 0,
    "sex": "Male", "cp": "asymptomatic", "restecg": "normal", "slope": "flat",
    "thal": "normal", "fbs": 0.0, "exang": 0.0,
}])[prep.FEATURE_COLUMNS]
cleaned = masker.transform(probe[num_cols].to_numpy(dtype=float))[0]
assert np.isnan(cleaned[1]) and np.isnan(cleaned[2]) and np.isnan(cleaned[4])
assert cleaned[0] == 50.0
print("[PASS] chol==0 / trestbps==0 / oldpeak<0 -> NaN; valid values preserved")

# id / dataset / num can never reach a model.
assert fitted.named_steps["preprocess"].remainder == "drop"
assert not any(t in n for n in feature_names for t in ("__id", "__dataset", "__num"))
print("[PASS] remainder='drop' and no id/dataset/num in transformed features")

## 9. Overall results

ROC-AUC and PR-AUC are primary because they are threshold-free. Accuracy is
secondary: the positive class is the majority at 55.3%, so a constant "always
disease" predictor already scores 0.553 accuracy.

In [ ]:
metric_table = format_metric_table(metrics_by_model)
metric_table

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4))
metric_table[["roc_auc", "pr_auc"]].plot.bar(ax=axes[0], color=["#4c72b0", "#55a868"])
axes[0].set_title("Threshold-free metrics")
axes[0].set_ylim(0.75, 1.0)
axes[0].tick_params(axis="x", rotation=15)

metric_table[["recall", "precision", "f1"]].plot.bar(
    ax=axes[1], color=["#4c72b0", "#dd8452", "#55a868"]
)
axes[1].set_title("Metrics at threshold 0.5")
axes[1].set_ylim(0.75, 1.0)
axes[1].tick_params(axis="x", rotation=15)

plt.tight_layout()
plt.show()

In [ ]:
cm = pd.DataFrame(
    {name: frame.to_numpy().ravel() for name, frame in confusion_by_model.items()},
    index=["TN", "FP", "FN", "TP"],
)
cm

fig, axes = plt.subplots(1, 3, figsize=(15, 4))
for ax, name in zip(axes, confusion_by_model):
    sns.heatmap(
        confusion_by_model[name].to_numpy(), annot=True, fmt="d", cmap="Blues",
        xticklabels=["pred no dz", "pred dz"], yticklabels=["true no dz", "true dz"], ax=ax,
    )
    ax.set_title(name)
plt.tight_layout()
plt.show()

### Reading the comparison honestly

No model is declared "best" here. The three models are within a few points of
each other on ROC-AUC (0.88-0.90), and with only 184 test rows those gaps are
inside the noise. What the table does support:

* All three learn real signal — ROC-AUC well above the 0.5 chance line.
* Logistic Regression is the strongest on both threshold-free metrics *and* the
  simplest. On a dataset this small and this noisy, that is not a surprise.
* XGBoost ranks last here despite being the most flexible model. Un-tuned boosting
  with 736 training rows is exactly the regime where it overfits the noise.
* All three make the same type of error: more false negatives than false
  positives, because the positive class is the majority.

A per-fold or repeated cross-validation estimate would be needed to say anything
confident about the small differences.

## 10. Cohort-wise results

Reported per source cohort, with a `reliable` flag. A cohort with fewer than 50
test rows, or fewer than 10 positives or negatives, gets `reliable = False`: the
figures exist but the uncertainty around them is too wide to act on.

In [ ]:
cohort_table = format_cohort_table(cohort_metrics_by_model)
cohort_table[["n", "n_positive", "prevalence", "roc_auc", "pr_auc",
              "recall", "precision", "f1", "accuracy", "reliable"]]

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 4))
pivot = cohort_table.reset_index().pivot(index="cohort", columns="model", values="roc_auc")
pivot.plot.bar(ax=axes[0])
axes[0].set_title("ROC-AUC by cohort")
axes[0].set_ylim(0.4, 1.0)
axes[0].tick_params(axis="x", rotation=20)

prev = df.groupby("dataset")["target"].mean() * 100
axes[1].bar(prev.index, prev.values, color="#c44e52")
axes[1].set_title("Base rate: % disease per cohort")
axes[1].set_ylabel("% with num > 0")
axes[1].tick_params(axis="x", rotation=20)
for container in axes[1].containers:
    axes[1].bar_label(container, fmt="%.1f")

plt.tight_layout()
plt.show()

The cohort picture matters more than the ranking does:

* **Cleveland and Hungary** are the only cohorts with enough test rows for a
  trustworthy figure. Both land around ROC-AUC 0.85-0.95.
* **Switzerland** (25 test rows, 23 of them positive) and **VA Long Beach**
  (40 rows, 30 positive) are flagged unreliable. Their base rates are 93.5% and
  75%, so almost everything is positive and ROC-AUC there is unstable — one
  XGBoost row lands at 0.50.
* The pooled ROC-AUC of ~0.89 blends cohorts with 36% to 94% base rates. It is a
  useful single number for this dataset, but it does not describe how the model
  will behave in any one of them.

## 11. Feature importance

Linear coefficients for Logistic Regression, impurity-based importances for the
tree models. Read as a description of what the models used, not as a causal claim
and not as a clinical finding.

In [ ]:
top_n = 15

lr_pipe = prep.build_pipeline(prep.make_logistic_regression()).fit(X_train, y_train)
lr_coef = pd.Series(
    lr_pipe.named_steps["model"].coef_[0], index=feature_names
).sort_values(key=abs, ascending=False)

rf_pipe = prep.build_pipeline(prep.make_random_forest()).fit(X_train, y_train)
rf_imp = pd.Series(
    rf_pipe.named_steps["model"].feature_importances_, index=feature_names
)

xgb_pipe = prep.build_pipeline(prep.make_xgboost()).fit(X_train, y_train)
xgb_imp = pd.Series(
    xgb_pipe.named_steps["model"].feature_importances_, index=feature_names
)

fig, axes = plt.subplots(1, 3, figsize=(17, 5))
for ax, series, title in [
    (axes[0], lr_coef, "Logistic Regression coefficients"),
    (axes[1], rf_imp.sort_values(ascending=False), "Random Forest importance"),
    (axes[2], xgb_imp.sort_values(ascending=False), "XGBoost importance"),
]:
    series.head(top_n).sort_values(ascending=(title.startswith("Logistic"))).plot.barh(ax=ax, color="#4c72b0")
    ax.set_title(f"{title} (top {top_n})")
plt.tight_layout()
plt.show()

## 12. Warnings and limitations

1. **Small dataset.** 920 rows, 736 for training. Every metric carries wide
   uncertainty, and the gaps between the three models are inside it.
2. **Single split.** One 80/20 holdout, no repeated runs or cross-validation. The
   cohort-stratified design is sound, but one draw is still one draw.
3. **Cohorts are not comparable.** Prevalence spans 36.2% to 93.5%. Pooled metrics
   average over populations that differ clinically, and per-cohort metrics are
   mostly unreliable for lack of rows.
4. **`ca` correlates 0.516 with `num`.** Both describe vessel narrowing, so this
   is genuine information rather than a copied label — but it does mean part of
   the model's apparent strength comes from an angiographic measurement that a
   screening setting would not have.
5. **No hyper-parameter tuning.** These are honest baselines by design. Tuning is
   the next stage, not this one.
6. **`chol == 0` recoding is an interpretation.** Reading those 172 rows as
   "not recorded" is well supported (they cluster entirely in two cohorts) but it
   is a judgement call, and it is documented rather than silent.
7. **No external validation.** Nothing here has been tested on data outside this
   file.
8. **Research prototype for risk assessment.** Not a medical device, not a
   diagnostic system. These numbers must not be presented as clinical capability.

## 13. Persisted artefacts

`src/train.py` writes the fitted pipelines and this report to `models/`:

| File | Contents |
|------|----------|
| `logistic_regression_pipeline.joblib` | preprocessing + model, one object |
| `random_forest_pipeline.joblib` | preprocessing + model, one object |
| `xgboost_pipeline.joblib` | preprocessing + model, one object |
| `baseline_metrics_overall.csv` | metric table above |
| `baseline_metrics_by_cohort.csv` | cohort table above |
| `split_summary.csv` | per-cohort row counts and base rates |
| `baseline_report.json` | data hash, feature list, hyper-parameters, metrics, environment, limitations |

Each `.joblib` holds preprocessing and model together, so inference cannot drift
from training. Re-running `python -m src.train` reproduces them bit-for-bit.

In [ ]:
print("final check — raw CSV unchanged:", prep.file_sha256() == prep.EXPECTED_SHA256)
print("expected:", prep.EXPECTED_SHA256)